# Animation Comparaison SeapoPym DAG vs Seapodym-LMTL (Pacifique)

Ce notebook génère une animation GIF comparant la biomasse entre :

1. **Référence** : Seapodym-LMTL
2. **DAG Transport** : SeapoPym avec transport
3. **DAG No-Transport** : SeapoPym sans transport

**Paramètres** :

-   Période : 3 dernières années (2002-2004)
-   Résolution temporelle : Moyenne hebdomadaire
-   Projection : PlateCarree centrée sur -150°
-   Extent : Pacifique complet [100°E, 280°E, -60°, 60°]
-   Colormap : viridis
-   vmin = 0, vmax = quantile 0.75 de la référence
-   Vitesse : 3 images/seconde


In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.animation import PillowWriter
import numpy as np
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point
from IPython.display import Image

# Chemins
DATA_DIR = "./data"
FIGURES_DIR = "./figures"

# Création du dossier figures s'il n'existe pas
os.makedirs(FIGURES_DIR, exist_ok=True)

# Entrées
FILE_REF = os.path.join(DATA_DIR, "seapodym_lmtl_output_pacific_ref.zarr")
FILE_TRANS = os.path.join(DATA_DIR, "seapopym_pacific_transport.zarr")
FILE_NO_TRANS = os.path.join(DATA_DIR, "seapopym_pacific_no_transport.zarr")

# Sortie
OUTPUT_GIF = os.path.join(FIGURES_DIR, "animation_pacific_weekly.gif")

print(f"Ref: {FILE_REF}")
print(f"Transport: {FILE_TRANS}")
print(f"No-Transport: {FILE_NO_TRANS}")

## 1. Chargement des Données

Les données sont chargées. La fonction `add_cyclic_point` de cartopy sera utilisée lors du plotting pour gérer automatiquement le wrapping autour de la ligne de changement de date.


In [ ]:
# Chargement Reference (Zooplankton)
ds_ref = xr.open_zarr(FILE_REF)
if "zooplankton" in ds_ref:
    ref = ds_ref["zooplankton"].load()
else:
    raise ValueError("Variable 'zooplankton' not found in reference file.")

# Chargement DAG Transport
ds_trans = xr.open_zarr(FILE_TRANS)
dag_trans = ds_trans["biomass"].load()

# Chargement DAG No-Transport
ds_no_trans = xr.open_zarr(FILE_NO_TRANS)
dag_no_trans = ds_no_trans["biomass"].load()

# Standardisation des noms de coordonnées
rename_dict = {"y": "latitude", "x": "longitude"}

ref = ref.rename({k: v for k, v in rename_dict.items() if k in ref.dims})
dag_trans = dag_trans.rename({k: v for k, v in rename_dict.items() if k in dag_trans.dims})
dag_no_trans = dag_no_trans.rename({k: v for k, v in rename_dict.items() if k in dag_no_trans.dims})

print("Données chargées.")
print(f"Ref shape: {ref.shape}")
print(f"Trans shape: {dag_trans.shape}")
print(f"No-Trans shape: {dag_no_trans.shape}")
print(f"\nLongitude range:")
print(f"  Ref: {ref.longitude.min().item():.2f} à {ref.longitude.max().item():.2f}")
print(
    f"  Transport: {dag_trans.longitude.min().item():.2f} à {dag_trans.longitude.max().item():.2f}"
)

## 2. Prétraitement

Période : 3 dernières années (2002-2004)  
Résolution temporelle : Moyenne hebdomadaire (1W)


In [ ]:
# Période de comparaison : 2002-2004
TIME_START = "2002-01-01"
TIME_END = "2004-12-31"

# Sélection temporelle
ref_cut = ref.sel(time=slice(TIME_START, TIME_END))
dag_trans_cut = dag_trans.sel(time=slice(TIME_START, TIME_END))
dag_no_trans_cut = dag_no_trans.sel(time=slice(TIME_START, TIME_END))

print(f"Avant resampling:")
print(f"  Référence: {len(ref_cut.time)} pas de temps")
print(f"  Transport: {len(dag_trans_cut.time)} pas de temps")
print(f"  No-Transport: {len(dag_no_trans_cut.time)} pas de temps")

# Resampling hebdomadaire (moyenne par semaine)
# Note: Les données SeapoPym sont 3-horaires, la référence est journalière
ref_weekly = ref_cut.resample(time="1W").mean()
dag_trans_weekly = dag_trans_cut.resample(time="1W").mean()
dag_no_trans_weekly = dag_no_trans_cut.resample(time="1W").mean()

print(f"\nAprès resampling hebdomadaire:")
print(f"  Référence: {len(ref_weekly.time)} semaines")
print(f"  Transport: {len(dag_trans_weekly.time)} semaines")
print(f"  No-Transport: {len(dag_no_trans_weekly.time)} semaines")

# Normalisation des timestamps à 00:00
ref_weekly["time"] = ref_weekly.time.dt.floor("D")
dag_trans_weekly["time"] = dag_trans_weekly.time.dt.floor("D")
dag_no_trans_weekly["time"] = dag_no_trans_weekly.time.dt.floor("D")

# Alignement des grilles
ref_weekly, dag_trans_weekly = xr.align(ref_weekly, dag_trans_weekly, join="inner")
_, dag_no_trans_weekly = xr.align(ref_weekly, dag_no_trans_weekly, join="inner")

print(f"\nAprès alignement:")
print(f"  Période alignée: {len(ref_weekly.time)} semaines")
print(f"  Grille alignée: {len(ref_weekly.latitude)} x {len(ref_weekly.longitude)}")

## 3. Calcul de vmax (quantile 0.75)


In [ ]:
# Calcul de vmax = quantile 0.75 de la référence sur toutes les dimensions
vmin = 0
vmax = ref_weekly.quantile(0.75).item()

print(f"Échelle de couleur:")
print(f"  vmin = {vmin}")
print(f"  vmax = {vmax:.2f} (quantile 0.75 de la référence)")

## 4. Création de l'Animation

Projection PlateCarree centrée sur -150° avec éléments cartographiques.


In [ ]:
# Nombre de frames
n_frames = len(ref_weekly.time)

print(f"Création de l'animation...")
print(f"  Nombre de frames: {n_frames}")
print(f"  Vitesse: 3 fps")
print(f"  Durée: {n_frames / 3:.1f} secondes")

# Configuration de la projection
proj = ccrs.PlateCarree(central_longitude=-150)
data_crs = ccrs.PlateCarree()

# Extent du Pacifique [lon_min, lon_max, lat_min, lat_max]
# Convention [0, 360] pour couvrir tout le Pacifique
extent = [100, 280, -60, 60]

# Création de la figure avec 3 subplots
fig, axes = plt.subplots(1, 3, figsize=(20, 6), subplot_kw={"projection": proj})

# Variable pour stocker la colorbar
cbar = None


# Fonction d'initialisation
def init():
    global cbar
    for ax in axes:
        ax.clear()
    cbar = None
    return []


# Fonction d'animation
def animate(frame):
    global cbar

    for ax in axes:
        ax.clear()

    # Récupération de la date
    time_pd = ref_weekly.time.to_series().iloc[frame]
    year = time_pd.year
    week = time_pd.isocalendar().week

    # Titre global
    fig.suptitle(
        f"Biomasse Zooplancton - Année {year} / Semaine {week}",
        fontsize=18,
        fontweight="bold",
        y=0.95,
    )

    # Données pour cette frame
    ref_frame = ref_weekly.isel(time=frame)
    trans_frame = dag_trans_weekly.isel(time=frame)
    no_trans_frame = dag_no_trans_weekly.isel(time=frame)

    # Liste pour stocker les images
    images = []
    titles = ["Référence (Seapodym-LMTL)", "DAG Transport", "DAG No-Transport"]
    data_frames = [ref_frame, trans_frame, no_trans_frame]

    for i, (ax, title, data) in enumerate(zip(axes, titles, data_frames)):
        # Configuration de la carte
        ax.set_extent(extent, crs=data_crs)

        # Ajout des éléments cartographiques
        ax.coastlines(resolution="110m", linewidth=0.5, color="black")
        ax.add_feature(cfeature.LAND, facecolor="lightgray", edgecolor="none", zorder=1)

        # Grille
        gl = ax.gridlines(draw_labels=False, linewidth=0.5, color="gray", alpha=0.5, linestyle="--")

        # Ajout du point cyclique pour fermer le gap autour de la ligne de changement de date
        data_cyclic, lon_cyclic = add_cyclic_point(data.values, coord=data.longitude.values)

        # Plot des données avec point cyclique
        im = ax.pcolormesh(
            lon_cyclic,
            data.latitude.values,
            data_cyclic,
            transform=data_crs,
            cmap="viridis",
            vmin=vmin,
            vmax=vmax,
            shading="auto",
            zorder=0,
        )

        ax.set_title(title, fontsize=13, fontweight="bold", pad=10)
        images.append(im)

    # Colorbar commune (créée une seule fois)
    if cbar is None:
        cbar = fig.colorbar(
            images[0], ax=axes, orientation="horizontal", pad=0.05, fraction=0.046, shrink=0.8
        )
        cbar.set_label("Biomasse (g/m²)", fontsize=12, fontweight="bold")

    plt.tight_layout(rect=[0, 0.05, 1, 0.92])

    return images


# Création de l'animation
anim = animation.FuncAnimation(
    fig,
    animate,
    init_func=init,
    frames=n_frames,
    interval=333,  # 333ms ≈ 3 fps
    blit=False,
)

print("Animation créée.")

## 5. Sauvegarde du GIF


In [ ]:
# Sauvegarde en GIF
print(f"Sauvegarde du GIF en cours (cela peut prendre plusieurs minutes)...")
writer = PillowWriter(fps=3)
anim.save(OUTPUT_GIF, writer=writer, dpi=150)

print(f"✅ Animation sauvegardée: {OUTPUT_GIF}")
print(f"   Durée: {n_frames / 3:.1f} secondes")
print(f"   Nombre de frames: {n_frames}")

# Taille du fichier
file_size_mb = os.path.getsize(OUTPUT_GIF) / (1024 * 1024)
print(f"   Taille: {file_size_mb:.1f} MB")